# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarahnjunge/starter-notebooks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/Sarahnjunge/starter-notebooks.git
%cd starter-notebooks
!ls data/raw

Cloning into 'starter-notebooks'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 163 (delta 69), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 1.89 MiB | 8.43 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/starter-notebooks
content_refresh_anonymized.csv


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*
I build the feature vector from the page, search, traffic, engagement, freshness, and ranking signals available in the dataset.

I exclude identifiers, the target, and the field used to derive the target. Numeric features will be handled separately from categorical features so that missing values can be filled appropriately before encoding.


In [4]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

target = "trend_direction"

excluded = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

feature_columns = [
    col for col in df.columns
    if col not in excluded
]

numeric_features = df[feature_columns].select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = df[feature_columns].select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Total feature columns:", len(feature_columns))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Total feature columns: 40
Numeric features: 29
Categorical features: 11

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical features:
['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


## 2. Feature notes (meaning, missing, categorical, available-when?)

For each candidate feature, I check its data type, missing values, and whether it would be available at the moment the prediction is made.

Numeric features will use median imputation for missing values. Categorical features will use `"missing"` as an explicit category. I also flag fields that may represent system or product information rather than signals about the content page itself.


In [5]:


feature_summary = pd.DataFrame({
    "feature": feature_columns,
    "dtype": df[feature_columns].dtypes.astype(str).values,
    "missing_count": df[feature_columns].isna().sum().values,
    "missing_pct": (
        df[feature_columns].isna().mean().values * 100
    ).round(2)
})

print(feature_summary.to_string(index=False))


               feature   dtype  missing_count  missing_pct
         search_volume float64           2468         8.23
           competition float64           2468         8.23
     competition_level  object           2610         8.70
                   cpc float64           2468         8.23
          content_type  object              0         0.00
           main_intent  object           2374         7.91
            word_count float64           7699        25.66
            char_count float64           7699        25.66
         provider_used  object          21438        71.46
            model_used  object           5733        19.11
       impressions_90d   int64              0         0.00
            clicks_90d   int64              0         0.00
         pageviews_90d   int64              0         0.00
          sessions_90d   int64              0         0.00
             users_90d   int64              0         0.00
  engaged_sessions_90d   int64              0         0.

### Feature availability and missing values

The numeric features are primarily historical search, traffic, engagement, ranking, content-age, and freshness measurements. Most of these fields are available from the historical data used to construct the prediction features.

Missing numeric values will be handled with median imputation, while missing categorical values will be represented as `"missing"` before one-hot encoding.

The largest missingness is in `provider_used` (71.46%), followed by `word_count` and `char_count` (25.66% each). `model_used` is missing in 19.11% of rows. These fields require additional leakage and usefulness checks before being included.

The historical performance fields such as impressions, clicks, sessions, CTR, engagement, and ranking are treated as available-before-prediction signals for this starter-data exercise. Because the starter dataset does not contain calendar dates, this availability assumption is a limitation that will be documented rather than presented as proof of production-time availability.


## 3. The leakage hunt
I test the candidate features for direct target leakage and suspicious fields.

The main leakage risk is `trend_pct`, because `trend_direction` is derived from it. Identifiers are also excluded because they identify the row rather than provide a legitimate predictive signal.

I also inspect fields that may encode product or system behavior, such as `provider_used` and `model_used`, because these may not represent information that should be used to make a page-level prediction.


In [6]:


check_columns = [
    "trend_pct",
    "trend_direction",
    "provider_used",
    "model_used",
    "content_id",
    "client_id"
]

for col in check_columns:
    print(f"\n--- {col} ---")
    print("dtype:", df[col].dtype)
    print("missing:", df[col].isna().sum())
    print("unique:", df[col].nunique(dropna=True))
    print(df[col].dropna().value_counts().head(10))



--- trend_pct ---
dtype: float64
missing: 3388
unique: 2712
trend_pct
-100.0    1395
 0.0       443
-50.0      281
-66.7      172
 100.0     145
-33.3      137
-75.0       98
-80.0       96
 50.0       88
-25.0       81
Name: count, dtype: int64

--- trend_direction ---
dtype: object
missing: 0
unique: 5
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

--- provider_used ---
dtype: object
missing: 21438
unique: 2
provider_used
google    7364
openai    1198
Name: count, dtype: int64

--- model_used ---
dtype: object
missing: 5733
unique: 5
model_used
gemini-3-flash-preview    13271
gpt-4o-mini                4981
gemini-2.5-flash           3665
gpt-5-mini                 1598
unknown                     752
Name: count, dtype: int64

--- content_id ---
dtype: object
missing: 0
unique: 30000
content_id
content_6880eb215048    1
content_fe5d259e6bc5    1
content_2dfd17269502    1
content_81a91fe32bc2    1
content_6f

In [7]:
# Final leakage and privacy/product-field check

excluded_for_leakage = [
    "trend_pct",
    "trend_direction",
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

final_features = [
    col for col in df.columns
    if col not in excluded_for_leakage
]

print("Excluded fields:")
for col in excluded_for_leakage:
    print("-", col)

print("\nFinal feature count:", len(final_features))
print("\nFinal features:")
print(final_features)

Excluded fields:
- trend_pct
- trend_direction
- content_id
- client_id
- provider_used
- model_used

Final feature count: 38

Final features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


## 4. What I excluded and why

* `trend_direction` — this is the prediction target, so it cannot be used as an input feature.
* `trend_pct` — this is directly related to how `trend_direction` is derived, creating target leakage.
* `content_id` — unique identifier with no generalizable page-level signal.
* `client_id` — identifies the client and could allow the model to memorize client-specific patterns instead of learning transferable signals.
* `provider_used` — mostly missing (71.46%) and represents a system/provider attribute rather than a page-level signal.
* `model_used` — represents the system/model used rather than an intrinsic property of the content page, so it was excluded to avoid learning product/system behavior.

The resulting feature vector contains 38 candidate features.


In [8]:


print("Final feature count:", len(final_features))
print("Final feature vector is ready.")

print("\nExcluded fields:")
print(excluded_for_leakage)

print("\nMissing values in final features:")
print(
    df[final_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

Final feature count: 38
Final feature vector is ready.

Excluded fields:
['trend_pct', 'trend_direction', 'content_id', 'client_id', 'provider_used', 'model_used']

Missing values in final features:
char_count           7699
word_count           7699
word_count_tier      7699
char_count_tier      7699
competition_level    2610
search_volume        2468
cpc                  2468
competition          2468
main_intent          2374
scroll_rate           125
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.